# 98 · Visual inspection

**Purpose:** Per-city visual overlays of reference vs candidate footprints for spot-checking.

**Inputs:** `data/01_raw/<city>/`, `outputs/metrics/`

**Outputs:** inspection figures

**Run order:** Optional.

**Last run:** _(fill in when you run it)_

In [ ]:
# Install dependencies not bundled with Colab by default.
# contextily fetches web map tiles (street map or satellite) for basemaps.
!pip install -q contextily

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 1 — Imports and config

In [ ]:
import gc
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box

try:
    import contextily as ctx
    HAS_CTX = True
except ImportError:
    HAS_CTX = False
    print("contextily not installed — run: pip install contextily")
    print("Basemap tiles will be skipped; install contextily to enable them.")

PROJECT_ROOT = Path('/content/drive/MyDrive/urban_validation')
DATA_DIR     = PROJECT_ROOT / 'data' / '01_raw'
OUT_DIR      = PROJECT_ROOT / 'outputs' / 'scratch' / 'visual_checks'
OUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = PROJECT_ROOT / 'outputs' / 'global_metrics' / 'vector_all_cities_merged.xlsx'
if not METRICS_PATH.exists():
    METRICS_PATH = PROJECT_ROOT / 'outputs' / 'vector_all_cities_merged.xlsx'

TRACKER_PATH  = PROJECT_ROOT / 'data' / '02_interim' / 'aoi_tracker.csv'
TARGET_CRS    = 'EPSG:3857'   # Web Mercator — global, required by contextily
LOW_F1_THRESH = 0.3

# Cap buildings per layer to avoid OOM on large cities (Cairo, Shanghai, etc.).
# For visual QA a random sample of 50k is indistinguishable from the full dataset.
MAX_BUILDINGS = 50_000

# Satellite imagery — makes spatial offsets immediately visible against real geography.
# Swap to ctx.providers.CartoDB.Positron for a lighter street-map basemap.
BASEMAP_SOURCE = ctx.providers.Esri.WorldImagery if HAS_CTX else None

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'DATA_DIR      : {DATA_DIR}  (exists={DATA_DIR.exists()})')
print(f'TRACKER_PATH  : {TRACKER_PATH}  (exists={TRACKER_PATH.exists()})')
print(f'METRICS_PATH  : {METRICS_PATH}  (exists={METRICS_PATH.exists()})')
print(f'Out dir       : {OUT_DIR}')
print(f'MAX_BUILDINGS : {MAX_BUILDINGS:,}')
print(f'Basemap       : {"enabled (" + str(BASEMAP_SOURCE.get("name", "")) + ")" if HAS_CTX else "disabled (contextily not installed)"}')

## Cell 2 — Load metrics and tracker

In [ ]:
# ── Metrics (vector_all_cities_merged.xlsx) ───────────────────────────────────
# If the file was not found at the standard path above, upload it here.
if not METRICS_PATH.exists():
    from google.colab import files as _colab_files
    import io
    print('METRICS_PATH not found — please upload vector_all_cities_merged.xlsx:')
    _up = _colab_files.upload()
    _buf = io.BytesIO(next(iter(_up.values())))
    metrics_raw = pd.read_excel(_buf, dtype=str)
else:
    metrics_raw = pd.read_excel(METRICS_PATH, dtype=str)

metrics_raw.columns = metrics_raw.columns.str.strip()

# Detect city and dataset columns robustly
_city_col    = next((c for c in metrics_raw.columns if c.lower() == 'city'), metrics_raw.columns[0])
_dataset_col = next((c for c in metrics_raw.columns if 'dataset' in c.lower() and c != _city_col), None)
_f1_col      = next((c for c in metrics_raw.columns if 'f1_city' in c.lower()), 
                    next((c for c in metrics_raw.columns if c.lower() == 'f1'), None))
_bias_col    = next((c for c in metrics_raw.columns if c.lower() == 'total_area_bias'), None)

print(f'Metrics columns: {list(metrics_raw.columns)}')
print(f'Using: city={_city_col!r}  dataset={_dataset_col!r}  f1={_f1_col!r}  bias={_bias_col!r}')

# Keep only Overture rows; index by city
_keep_cols = [_city_col] + [c for c in [_f1_col, _bias_col] if c is not None]
if _dataset_col:
    ov_rows = metrics_raw[metrics_raw[_dataset_col].str.strip().str.lower() == 'overture']
else:
    ov_rows = metrics_raw   # single-dataset file

ov_metrics = ov_rows[_keep_cols].copy()
ov_metrics = ov_metrics.rename(columns={_city_col: 'city', _f1_col: 'f1_city', _bias_col: 'total_area_bias'})
for col in ['f1_city', 'total_area_bias']:
    if col in ov_metrics.columns:
        ov_metrics[col] = pd.to_numeric(ov_metrics[col], errors='coerce')
ov_metrics = ov_metrics.set_index('city')
print(f'\nOverture metrics: {len(ov_metrics)} cities')

# ── AOI tracker ───────────────────────────────────────────────────────────────
tracker = pd.read_csv(TRACKER_PATH, dtype=str)
tracker.columns = tracker.columns.str.strip()
tracker = tracker.apply(lambda c: c.str.strip() if c.dtype == object else c)

# Filter to suitable cities only
_suitable_col = next((c for c in tracker.columns if 'suitable' in c.lower()), None)
if _suitable_col:
    tracker = tracker[tracker[_suitable_col].str.lower() == 'yes'].copy()

# Identify key columns — confirmed from src/utils/aoi_inventory.py
_id_col     = 'Dataset code'          # city slug used as unique ID (e.g. 'ssd-juba')
_folder_col = 'dataset_folder_name'   # disk folder name under data/01_raw/
_aoi_col    = 'aoi_file_name'         # AOI filename(s), pipe-separated for multi-AOI cities
# Reference file column: detected by 'reference' + 'file' in name
_ref_col    = next(
    (c for c in tracker.columns if 'reference' in c.lower() and 'file' in c.lower()),
    None,
)

print(f'\nTracker columns : {tracker.columns.tolist()}')
print(f'Using: id={_id_col!r}  folder={_folder_col!r}  aoi={_aoi_col!r}  ref={_ref_col!r}')
print(f'Cities in tracker (suitable): {tracker[_id_col].nunique()}')

## Cell 3 — Helper functions

In [ ]:
def _load_vector(path: Path, max_rows: int | None = None) -> gpd.GeoDataFrame | None:
    """Load a vector file keeping only geometry to minimise RAM.

    For parquet files the geometry column is read directly (skipping all
    attribute columns). For other formats the first max_rows rows are read.
    Sampling happens at load time — before any reprojection — so the peak
    in-memory size never exceeds max_rows geometries.
    """
    if not path.exists():
        return None
    try:
        if path.suffix.lower() == '.parquet':
            gdf = gpd.read_parquet(path, columns=['geometry'])
            if max_rows is not None and len(gdf) > max_rows:
                gdf = gdf.sample(max_rows, random_state=42).reset_index(drop=True)
        else:
            # rows= reads only the first N records without loading the full file
            kwargs = {'rows': max_rows} if max_rows is not None else {}
            gdf = gpd.read_file(path, **kwargs)
        return gdf
    except Exception as exc:
        print(f'    [WARN] could not load {path.name}: {exc}')
        return None


def load_aoi(folder_path: Path, aoi_spec: str) -> gpd.GeoDataFrame | None:
    """Load and dissolve AOI polygons (pipe-separated filenames, EPSG:4326)."""
    aoi_dir = folder_path / 'aoi'
    parts = []
    for name in str(aoi_spec).split('|'):
        name = name.strip()
        if not name:
            continue
        gdf = _load_vector(aoi_dir / name)   # AOIs are tiny — no cap needed
        if gdf is not None:
            parts.append(gdf)

    if not parts:
        return None

    combined = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
    if combined.crs is None:
        combined = combined.set_crs('EPSG:4326')
    return combined.dissolve().reset_index(drop=True).to_crs('EPSG:4326')


def load_reference(folder_path: Path, ref_spec: str | None) -> gpd.GeoDataFrame | None:
    """Load reference footprints, capped at MAX_BUILDINGS geometries."""
    vec_dir = folder_path / 'vector'
    if not vec_dir.exists():
        return None

    candidates = []
    if ref_spec and str(ref_spec).strip() not in ('', 'nan', 'None'):
        for name in str(ref_spec).split('|'):
            name = name.strip()
            if name:
                candidates.append(vec_dir / name)
    else:
        for pat in ('*_ref.*', '*_reference.*', '*hotosm*', '*worldbank*', '*sn7*'):
            candidates.extend(vec_dir.glob(pat))
        candidates = [
            p for p in candidates
            if not any(k in p.name.lower() for k in ('overture', 'gba', 'globfp', 'obt', 'tempo', 'wsf'))
        ]

    parts = []
    for path in candidates:
        gdf = _load_vector(path, max_rows=MAX_BUILDINGS)
        if gdf is not None:
            parts.append(gdf)

    if not parts:
        return None

    combined = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
    if combined.crs is None:
        combined = combined.set_crs('EPSG:4326')
    return combined


def load_overture(folder_path: Path, slug: str) -> gpd.GeoDataFrame | None:
    """Load Overture building footprints, capped at MAX_BUILDINGS geometries."""
    vec_dir = folder_path / 'vector'
    if not vec_dir.exists():
        return None

    preferred = vec_dir / f'{slug}_overture_building.parquet'
    if preferred.exists():
        candidates = [preferred]
    else:
        candidates = sorted(vec_dir.glob(f'{slug}_overture*.parquet'))
        if not candidates:
            alt_slug = folder_path.name.replace('-', '_')
            candidates = sorted(vec_dir.glob(f'{alt_slug}_overture*.parquet'))

    parts = [_load_vector(p, max_rows=MAX_BUILDINGS) for p in candidates]
    parts = [g for g in parts if g is not None]
    if not parts:
        return None

    combined = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
    if combined.crs is None:
        combined = combined.set_crs('EPSG:4326')
    return combined


def make_city_figure(city, aoi, ref, overture, f1, bias, has_aoi):
    """Overlay reference (red) and Overture (blue) footprints on a satellite basemap."""
    fig, ax = plt.subplots(figsize=(10, 10))

    if aoi      is not None: aoi.boundary.plot(ax=ax, color='black', linewidth=1.0)
    if ref      is not None and len(ref)      > 0: ref.boundary.plot(ax=ax, color='red',  linewidth=0.5)
    if overture is not None and len(overture) > 0: overture.boundary.plot(ax=ax, color='blue', linewidth=0.5)

    # Set extent using total_bounds (envelope only — no expensive unary_union).
    # Must happen BEFORE add_basemap so contextily fetches the right tiles.
    _layers = [g for g in (aoi, ref, overture) if g is not None and len(g) > 0]
    if _layers:
        all_bounds = np.array([g.total_bounds for g in _layers])
        minx, miny = all_bounds[:, 0].min(), all_bounds[:, 1].min()
        maxx, maxy = all_bounds[:, 2].max(), all_bounds[:, 3].max()
        pad = max(maxx - minx, maxy - miny) * 0.05
        ax.set_xlim(minx - pad, maxx + pad)
        ax.set_ylim(miny - pad, maxy + pad)

    if HAS_CTX and BASEMAP_SOURCE is not None:
        try:
            ctx.add_basemap(ax, crs=TARGET_CRS, source=BASEMAP_SOURCE, zoom='auto')
        except Exception:
            pass   # offline or tile server error — continue without basemap

    prefix   = '⚠ NO AOI  ' if not has_aoi else ''
    bias_str = f'{bias:+.1%}' if pd.notna(bias) else 'N/A'
    f1_str   = f'{f1:.3f}'    if pd.notna(f1)   else 'N/A'
    n_ref    = len(ref)      if ref      is not None else 0
    n_ov     = len(overture) if overture is not None else 0
    ax.set_title(
        f'{prefix}{city}  |  F1={f1_str}  |  Bias={bias_str}  |  '
        f'ref={n_ref:,}  overture={n_ov:,}',
        fontsize=11,
    )
    ax.set_axis_off()
    ax.legend(handles=[
        mpatches.Patch(edgecolor='black', facecolor='none', label='AOI'),
        mpatches.Patch(edgecolor='red',   facecolor='none', label='Reference'),
        mpatches.Patch(edgecolor='blue',  facecolor='none', label='Overture'),
    ], loc='lower right', fontsize=9)
    plt.tight_layout()
    return fig


print('Helper functions defined.')
print(f'Basemap : {"Esri WorldImagery (satellite)" if HAS_CTX else "disabled — pip install contextily"}')
print(f'Building cap : {MAX_BUILDINGS:,} per layer (sampled at load time)')

## Cell 4 — Main loop

In [ ]:
overview_rows = []

city_groups = tracker.groupby(_id_col, sort=False)
total = city_groups.ngroups
print(f'Processing {total} cities...  (buildings capped at {MAX_BUILDINGS:,} per layer, geometry-only load)\n')

for idx, (city, group) in enumerate(city_groups, 1):
    city        = str(city).strip()
    row0        = group.iloc[0]
    folder_name = str(row0.get(_folder_col, city)).strip()
    folder_path = DATA_DIR / folder_name
    slug        = folder_name.replace('-', '_').replace(' ', '_')

    aoi = ref = overture = fig = None   # ensure names exist for finally block

    try:
        aoi_spec = '|'.join(
            part.strip()
            for raw in group[_aoi_col].fillna('')
            for part in str(raw).split('|')
            if part.strip()
        )

        ref_spec = None
        if _ref_col:
            ref_specs = group[_ref_col].dropna().tolist()
            if ref_specs:
                ref_spec = '|'.join(
                    part.strip()
                    for raw in ref_specs
                    for part in str(raw).split('|')
                    if part.strip() not in ('', 'nan', 'None')
                ) or None

        # Load — cap and geometry-only sampling happens inside the helpers
        aoi      = load_aoi(folder_path, aoi_spec) if aoi_spec else None
        ref      = load_reference(folder_path, ref_spec)
        overture = load_overture(folder_path, slug)

        # Reproject to TARGET_CRS (only 50k geometries per layer at this point)
        if aoi      is not None: aoi      = aoi.to_crs(TARGET_CRS)
        if ref      is not None: ref      = ref.to_crs(TARGET_CRS)
        if overture is not None: overture = overture.to_crs(TARGET_CRS)

        f1   = float(ov_metrics.loc[city, 'f1_city'])             if city in ov_metrics.index and 'f1_city'             in ov_metrics.columns else float('nan')
        bias = float(ov_metrics.loc[city, 'total_area_bias']) if city in ov_metrics.index and 'total_area_bias' in ov_metrics.columns else float('nan')

        has_aoi      = aoi      is not None and len(aoi)      > 0
        has_ref      = ref      is not None and len(ref)      > 0
        has_overture = overture is not None and len(overture) > 0

        if not has_ref and not has_overture:
            flag = 'NO DATA'
            print(f'[{idx:3d}/{total}] [SKIP] {city}: no ref or overture data')
            overview_rows.append(dict(city=city, f1=f1, signed_area_bias=bias,
                                      has_aoi=has_aoi, has_ref=False, has_overture=False, flag=flag))
            continue

        flags = []
        if not has_aoi:                          flags.append('NO AOI')
        if not has_ref:                          flags.append('NO REF')
        if not has_overture:                     flags.append('NO OVERTURE')
        if pd.notna(f1) and f1 < LOW_F1_THRESH: flags.append('LOW F1')
        flag = ' | '.join(flags) if flags else 'OK'

        fig = make_city_figure(city, aoi, ref, overture, f1, bias, has_aoi)
        fig.savefig(OUT_DIR / f'{city}_overture_vs_ref.png', dpi=150, bbox_inches='tight')
        f1_str = f'{f1:.3f}' if pd.notna(f1) else 'N/A'
        print(f'[{idx:3d}/{total}] [OK]   {city:<30}  F1={f1_str}  flag={flag}')

    except Exception as exc:
        flag = f'ERROR: {exc}'
        print(f'[{idx:3d}/{total}] [ERR]  {city}: {exc}')
        f1 = bias = float('nan')
        has_aoi = has_ref = has_overture = False

    finally:
        # Free all geometry data after every city — prevents RAM accumulation.
        plt.close('all')
        del aoi, ref, overture, fig
        gc.collect()

    overview_rows.append(dict(city=city, f1=f1, signed_area_bias=bias,
                              has_aoi=has_aoi, has_ref=has_ref, has_overture=has_overture,
                              flag=flag))

overview_df = pd.DataFrame(overview_rows).sort_values('f1')
overview_df.to_csv(OUT_DIR / '_overview.csv', index=False)
print(f'\nDone. {len(overview_rows)} cities processed.')
print(overview_df['flag'].value_counts().to_string())

## Cell 5 — Summary

In [ ]:
print('=== Cities flagged as LOW F1 (< {:.0%}) ==='.format(LOW_F1_THRESH))
_low = overview_df[overview_df['flag'].str.contains('LOW F1', na=False)]
if _low.empty:
    print('  None.')
else:
    print(_low[['city', 'f1', 'signed_area_bias', 'flag']].to_string(index=False))

print('\n=== Cities with missing AOI ===')
_no_aoi = overview_df[~overview_df['has_aoi']]
if _no_aoi.empty:
    print('  None.')
else:
    print(_no_aoi[['city', 'f1', 'flag']].to_string(index=False))

print('\n=== Cities with missing reference data ===')
_no_ref = overview_df[~overview_df['has_ref']]
if _no_ref.empty:
    print('  None.')
else:
    print(_no_ref[['city', 'f1', 'flag']].to_string(index=False))

print('\n=== Cities with missing Overture data ===')
_no_ov = overview_df[~overview_df['has_overture']]
if _no_ov.empty:
    print('  None.')
else:
    print(_no_ov[['city', 'f1', 'flag']].to_string(index=False))

print('\n=== Cities with errors ===')
_errors = overview_df[overview_df['flag'].str.startswith('ERROR', na=False)]
if _errors.empty:
    print('  None.')
else:
    print(_errors[['city', 'flag']].to_string(index=False))

print(f'\nPNGs saved to: {OUT_DIR}')
print(f'Overview CSV : {OUT_DIR / "_overview.csv"}')